# Single-Ticker Comparison: 5 Model Families

This notebook compares 5 model families for each ticker separately.

Tickers:
- `TSLA`
- `AAPL`

Feature sets:
- `price_only`
- `price_plus_alt`

Model families:
- `logreg`
- `svm_rbf`
- `random_forest`
- `hist_gradient_boosting`
- `mlp_small`

Artifacts are written to `notebooks_two_tickers/outputs/ticker_family_comparison`.


In [1]:
import pandas as pd

from ticker_family_comparison_experiment import run_ticker_family_comparison_experiment

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 260)


In [2]:
result = run_ticker_family_comparison_experiment()

print(f"Dataset: {result['dataset_path']}")
print(f"Metrics: {result['metrics_path']}")
print(f"Predictions: {result['predictions_path']}")
print(f"Metadata: {result['metadata_path']}")
print(f"Feature sets: {result['feature_sets_path']}")


Dataset: C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\notebooks_two_tickers\outputs\datasets\stock_direction_two_tickers_base.csv
Metrics: C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\notebooks_two_tickers\outputs\ticker_family_comparison\metrics.csv
Predictions: C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\notebooks_two_tickers\outputs\ticker_family_comparison\predictions.csv
Metadata: C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\notebooks_two_tickers\outputs\ticker_family_comparison\run_metadata.json
Feature sets: C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\notebooks_two_tickers\outputs\ticker_family_comparison\feature_sets.json


In [3]:
test_metrics = result["metrics"].query("split == 'test'").copy()
test_metrics = test_metrics.sort_values(["ticker", "model_name", "feature_set"]).reset_index(drop=True)
test_metrics


,ticker,model_name,feature_set,split,n_rows,n_features,accuracy,balanced_accuracy,f1,roc_auc
0,AAPL,hist_gradient_boosting,price_only,test,128,14,0.546875,0.533990,0.618421,0.552709
1,AAPL,hist_gradient_boosting,price_plus_alt,test,128,32,0.507812,0.505665,0.540146,0.508867
2,AAPL,logreg,price_only,test,128,14,0.507812,0.511576,0.511628,0.497044
3,AAPL,logreg,price_plus_alt,test,128,32,0.500000,0.508867,0.475410,0.511576
4,AAPL,mlp_small,price_only,test,128,14,0.562500,0.532020,0.681818,0.544828
5,AAPL,mlp_small,price_plus_alt,test,128,32,0.468750,0.459606,0.534247,0.500739
6,AAPL,random_forest,price_only,test,128,14,0.507812,0.505665,0.540146,0.538177
7,AAPL,random_forest,price_plus_alt,test,128,32,0.476562,0.480049,0.480620,0.515271
8,AAPL,svm_rbf,price_only,test,128,14,0.507812,0.467241,0.666667,0.475000
9,AAPL,svm_rbf,price_plus_alt,test,128,32,0.546875,0.501478,0.704082,0.475862


In [4]:
summary = test_metrics.pivot_table(
    index=["ticker", "model_name"],
    columns="feature_set",
    values=["balanced_accuracy", "roc_auc"],
)
summary


balanced_accuracy                   roc_auc               
feature_set                          price_only price_plus_alt price_only price_plus_alt
ticker model_name                                                                       
AAPL   hist_gradient_boosting          0.533990       0.505665   0.552709       0.508867
       logreg                          0.511576       0.508867   0.497044       0.511576
       mlp_small                       0.532020       0.459606   0.544828       0.500739
       random_forest                   0.505665       0.480049   0.538177       0.515271
       svm_rbf                         0.467241       0.501478   0.475000       0.475862
TSLA   hist_gradient_boosting          0.581675       0.534512   0.569885       0.533039
       logreg                          0.609555       0.484893   0.618767       0.513879
       mlp_small                       0.639646       0.480103   0.650209       0.495456
       random_forest                   0.625276       0.587693   0.611152       0.566200
       svm_rbf                         0.493982       0.562024   0.607222       0.566323

In [5]:
price = test_metrics[test_metrics["feature_set"] == "price_only"].copy()
alt = test_metrics[test_metrics["feature_set"] == "price_plus_alt"].copy()
delta = price.merge(
    alt,
    on=["ticker", "model_name", "split"],
    suffixes=("_price", "_price_alt"),
)
delta["delta_balanced_accuracy"] = delta["balanced_accuracy_price_alt"] - delta["balanced_accuracy_price"]
delta["delta_roc_auc"] = delta["roc_auc_price_alt"] - delta["roc_auc_price"]
delta[["ticker", "model_name", "delta_balanced_accuracy", "delta_roc_auc"]].sort_values(["ticker", "model_name"]).reset_index(drop=True)


,ticker,model_name,delta_balanced_accuracy,delta_roc_auc
0,AAPL,hist_gradient_boosting,-0.028325,-0.043842
1,AAPL,logreg,-0.002709,0.014532
2,AAPL,mlp_small,-0.072414,-0.044089
3,AAPL,random_forest,-0.025616,-0.022906
4,AAPL,svm_rbf,0.034236,0.000862
5,TSLA,hist_gradient_boosting,-0.047163,-0.036846
6,TSLA,logreg,-0.124662,-0.104888
7,TSLA,mlp_small,-0.159543,-0.154753
8,TSLA,random_forest,-0.037583,-0.044952
9,TSLA,svm_rbf,0.068042,-0.040899
